# 03 — Business Analytics Demo

This notebook runs all four business analytics pipelines against the generated Amazon fixture data:

1. **RFM Analysis** — Customer segmentation (Recency / Frequency / Monetary)
2. **Geo Analytics** — Revenue aggregation by country and region
3. **Campaign Optimizer** — iPhone 17 marketing KPI analysis
4. **Customer Intent Scoring** — Purchase intent prediction

> ⚙️ **Prerequisites**:
> - Run `make generate-fixtures` to create the Parquet sample data
> - Java 17 installed for PySpark

## 1. ⚙️ Setup

In [ ]:
import sys
import warnings
from pathlib import Path
import tempfile

sys.path.insert(0, '../../src')
warnings.filterwarnings('ignore')

# Output directory for Delta tables
OUTPUT_DIR = tempfile.mkdtemp()
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
from etl_agent.spark.session import get_or_create_spark
from etl_agent.spark.optimizer import apply_all_optimizations

spark = get_or_create_spark(app_name='BusinessAnalyticsDemo')
apply_all_optimizations(spark)
print(f'Spark {spark.version} ready')

## 2. Load Fixture Data

The fixtures are generated by `scripts/generate_fixtures.py` (run `make generate-fixtures`).

In [ ]:
FIXTURES_DIR = Path('../../tests/fixtures')

# Check fixtures exist
for name in ['amazon_customers', 'amazon_orders', 'amazon_campaigns', 'amazon_products']:
    path = FIXTURES_DIR / f'{name}.parquet'
    exists = path.exists()
    print(f'  {"✅" if exists else "❌"} {name}.parquet')

print()
print('If any are missing, run: make generate-fixtures')

In [ ]:
# Load fixtures
try:
    customers_df = spark.read.parquet(str(FIXTURES_DIR / 'amazon_customers.parquet'))
    orders_df    = spark.read.parquet(str(FIXTURES_DIR / 'amazon_orders.parquet'))
    campaigns_df = spark.read.parquet(str(FIXTURES_DIR / 'amazon_campaigns.parquet'))
    products_df  = spark.read.parquet(str(FIXTURES_DIR / 'amazon_products.parquet'))

    print(f'Customers : {customers_df.count():,} rows')
    print(f'Orders    : {orders_df.count():,} rows')
    print(f'Campaigns : {campaigns_df.count():,} rows')
    print(f'Products  : {products_df.count():,} rows')
except Exception as e:
    print(f'⚠️  Could not load fixtures: {e}')
    print('   Run: make generate-fixtures')

## 3. RFM Customer Segmentation

In [ ]:
from etl_agent.analytics.rfm_analysis import run_rfm_analysis

rfm_output = f'{OUTPUT_DIR}/rfm_scores'

run_rfm_analysis(spark, orders_df, output_path=rfm_output)

rfm_df = spark.read.format('delta').load(rfm_output)
print(f'RFM table: {rfm_df.count():,} customers scored')
rfm_df.show(5)

In [ ]:
# Segment distribution
from pyspark.sql import functions as F

print('=== RFM Segment Distribution ===')
rfm_df.groupBy('rfm_segment').agg(
    F.count('customer_id').alias('customer_count'),
    F.round(F.avg('monetary'), 2).alias('avg_spend'),
    F.round(F.avg('frequency'), 1).alias('avg_orders'),
).orderBy(F.col('customer_count').desc()).show(truncate=False)

In [ ]:
# Visualize segment sizes
try:
    import matplotlib.pyplot as plt
    
    segments = rfm_df.groupBy('rfm_segment').count().toPandas()
    segments = segments.sort_values('count', ascending=True)
    
    colors = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#3b82f6']
    
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(segments['rfm_segment'], segments['count'], color=colors[:len(segments)])
    ax.set_xlabel('Customer Count')
    ax.set_title('RFM Customer Segments')
    
    for bar, val in zip(bars, segments['count']):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping visualization')

## 4. Geographic Revenue Analytics

In [ ]:
from etl_agent.analytics.geo_analytics import run_geo_analysis

geo_output = f'{OUTPUT_DIR}/geo_revenue'

run_geo_analysis(spark, orders_df, customers_df, output_path=geo_output)

geo_df = spark.read.format('delta').load(geo_output)
print(f'Geo table: {geo_df.count():,} country/region combinations')
print('\nTop 10 regions by revenue:')
geo_df.orderBy(F.col('total_revenue').desc()).show(10, truncate=False)

In [ ]:
# Top 5 countries by total revenue
print('=== Top 5 Countries by Revenue ===')
geo_df.groupBy('country').agg(
    F.sum('total_revenue').alias('country_revenue'),
    F.sum('unique_customers').alias('total_customers'),
).orderBy(F.col('country_revenue').desc()).show(5, truncate=False)

## 5. iPhone 17 Campaign Optimizer

In [ ]:
from etl_agent.analytics.campaign_optimizer import run_campaign_analysis

campaign_output = f'{OUTPUT_DIR}/campaign_kpis'

run_campaign_analysis(spark, campaigns_df, output_path=campaign_output)

campaign_df = spark.read.format('delta').load(campaign_output)
print(f'Campaign KPIs: {campaign_df.count()} iPhone 17 campaigns analysed')
campaign_df.show(truncate=False)

In [ ]:
# Summary KPIs
print('=== iPhone 17 Campaign Summary ===')
campaign_df.agg(
    F.round(F.avg('conversion_rate'), 4).alias('avg_conversion_rate'),
    F.round(F.avg('roi_pct'), 2).alias('avg_roi_pct'),
    F.round(F.avg('revenue_per_impression'), 4).alias('avg_rev_per_impression'),
).show(truncate=False)

print('Campaign Grade Breakdown:')
campaign_df.groupBy('campaign_grade').count().orderBy('campaign_grade').show()

## 6. Customer Intent Scoring

In [ ]:
from etl_agent.analytics.customer_intent import run_intent_scoring

intent_output = f'{OUTPUT_DIR}/intent_scores'

run_intent_scoring(spark, orders_df, customers_df, output_path=intent_output)

intent_df = spark.read.format('delta').load(intent_output)
print(f'Intent scores: {intent_df.count():,} customers scored')
intent_df.show(5)

In [ ]:
# Intent segment distribution
print('=== Customer Intent Distribution ===')
intent_df.groupBy('intent_segment').agg(
    F.count('customer_id').alias('customer_count'),
    F.round(F.avg('intent_score'), 1).alias('avg_score'),
    F.round(F.sum('total_spent'), 0).alias('total_revenue'),
).orderBy(F.col('avg_score').desc()).show(truncate=False)

## 7. Cross-Pipeline Insights

In [ ]:
# Join RFM + Intent for a combined view
combined = rfm_df.join(intent_df.select('customer_id', 'intent_score', 'intent_segment'),
                       on='customer_id', how='inner')

print('=== RFM Segment × Intent Segment Cross-Analysis ===')
combined.groupBy('rfm_segment', 'intent_segment').agg(
    F.count('customer_id').alias('count'),
    F.round(F.avg('monetary'), 0).alias('avg_spend'),
).orderBy('rfm_segment', 'intent_segment').show(20, truncate=False)

In [ ]:
# High-value targets: Champions with High Intent
high_value_targets = combined.filter(
    (F.col('rfm_segment') == 'Champions') &
    (F.col('intent_segment') == 'High Intent')
)

print(f'🎯 High-value targets (Champions × High Intent): {high_value_targets.count():,} customers')
print('These are your best candidates for iPhone 17 targeted campaigns!')
high_value_targets.select('customer_id', 'rfm_segment', 'intent_segment', 'monetary', 'intent_score') \
                  .orderBy(F.col('monetary').desc()).show(10)

## 8. Results Summary

In [ ]:
print('=' * 60)
print('  BUSINESS ANALYTICS PIPELINE RESULTS SUMMARY')
print('=' * 60)
print()
print(f'  📊 RFM Analysis')
print(f'     Total customers scored : {rfm_df.count():,}')
print(f'     Segments identified     : {rfm_df.select("rfm_segment").distinct().count()}')
print()
print(f'  🌍 Geo Analytics')
print(f'     Country/regions analysed: {geo_df.count():,}')
top_country = geo_df.groupBy("country").agg(F.sum("total_revenue").alias("rev")).orderBy(F.col("rev").desc()).first()
print(f'     Top country by revenue  : {top_country["country"]} (${top_country["rev"]:,.0f})')
print()
print(f'  📱 Campaign Optimizer')
print(f'     iPhone 17 campaigns     : {campaign_df.count()}')
grade_a = campaign_df.filter(F.col("campaign_grade") == "A").count()
print(f'     Grade A campaigns       : {grade_a}')
print()
print(f'  🎯 Customer Intent')
print(f'     Customers scored        : {intent_df.count():,}')
high_intent = intent_df.filter(F.col("intent_segment") == "High Intent").count()
print(f'     High Intent customers   : {high_intent:,}')
print()
print('=' * 60)

In [ ]:
spark.stop()
print('SparkSession stopped. Demo complete! 🎉')